<a href="https://colab.research.google.com/github/minaGalil/Imperial-Capstone/blob/main/Capstone_Week_4_Calling_Functions_ipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import pandas as pd
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, RBF, WhiteKernel
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor

np.random.seed(42)

# =========================================================
# WEEK DATA (FROM YOUR FILES)
# =========================================================

# Week 1 inputs
week1_inputs = [
    [0.761024, 0.713000],
    [0.732637, 0.906564],
    [0.522581, 0.591593, 0.350176],
    [0.564020, 0.473834, 0.390972, 0.258427],
    [0.196777, 0.892275, 0.855813, 0.891829],
    [0.728785, 0.146949, 0.767044, 0.739699, 0.020789],
    [0.016700, 0.532618, 0.280647, 0.222769, 0.407487, 0.748018],
    [0.035844, 0.064098, 0.010711, 0.024463, 0.361511, 0.783344, 0.515157, 0.880442],
]

# Week 1 outputs (correct row)
week1_outputs = [
    -7.565331332744927e-18,
    0.5530064492925906,
    -0.03333218343962805,
    -4.83054096204485,
    1257.680268889983,
    -0.8072367077314392,
    1.2394822144938658,
    9.5430069620611,
]

# Week 2 inputs
week2_inputs = [
    [0.751825, 0.811946],
    [0.980817, 0.626715],
    [0.996446, 0.737055, 0.850924],
    [0.404477, 0.413254, 0.303108, 0.434359],
    [0.521278, 0.505138, 0.985473, 0.994545],
    [0.341155, 0.021278, 0.626487, 0.970661, 0.032762],
    [0.431390, 0.173879, 0.071339, 0.124473, 0.403592, 0.624180],
    [0.064214, 0.412793, 0.081589, 0.008195, 0.974438, 0.216196, 0.139173, 0.110624],
]

# Week 2 outputs
week2_outputs = [
    2.495129202697582e-35,
    0.06646691679004207,
    -0.04126707799435369,
    -0.7158150747637886,
    1769.2992166742467,
    -0.721361811601727,
    1.493591747104962,
    9.7741312776374,
]

# Week 3 inputs
week3_inputs = [
    [0.702630, 0.955980],
    [0.700329, 1.000000],
    [0.977206, 0.330727, 0.000017],
    [0.393932, 0.393230, 0.377559, 0.426151],
    [0.449325, 0.603715, 1.000000, 1.000000],
    [0.941155, 0.654101, 0.079248, 0.961137, 0.229746],
    [0.153100, 0.236737, 0.190018, 0.000000, 0.365128, 0.744155],
    [0.103762, 0.017606, 0.204751, 0.194101, 0.745367, 0.674182, 0.125362, 0.044024],
]

# Week 3 outputs (latest row)
week3_outputs = [
    -8.189634555426656e-79,
    0.5980717829505922,
    -0.12802055717541724,
    0.2735224699683667,
    2027.715300336871,
    -1.6878746934171067,
    1.120167075371798,
    9.8898511444579,
]

# =========================================================
# LOOP
# =========================================================
results = []

for i in range(8):

    print("\n==============================")
    print(f"Function {i+1}")
    print("==============================")

    X = np.load(f"function{i+1}_inputs.npy")
    y = np.load(f"function{i+1}_outputs.npy")

    # Append all previous weeks
    for p, o in [
        (week1_inputs[i], week1_outputs[i]),
        (week2_inputs[i], week2_outputs[i]),
        (week3_inputs[i], week3_outputs[i]),
    ]:
        p = np.array(p)

        if not np.any(np.all(np.isclose(X, p), axis=1)):
            X = np.vstack([X, p])
            y = np.append(y, o)

    dim = X.shape[1]

    best_idx = np.argmax(y)
    best_input = X[best_idx]

    # =====================================================
    # MODELS
    # =====================================================

    # GP
    gp = GaussianProcessRegressor(normalize_y=True)
    gp.fit(X, y)

    # SVM
    threshold = np.quantile(y, 0.7)
    labels = (y >= threshold).astype(int)

    if len(np.unique(labels)) > 1:
        svm = Pipeline([
            ("scaler", StandardScaler()),
            ("svc", SVC(kernel="rbf", probability=True))
        ])
        svm.fit(X, labels)
        use_svm = True
    else:
        use_svm = False

    # Neural Network
    nn = Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(hidden_layer_sizes=(64,32), max_iter=500))
    ])
    nn.fit(X, y)

    # =====================================================
    # CANDIDATES
    # =====================================================
    candidates = np.random.uniform(0,1,(10000,dim))
    local = np.clip(best_input + np.random.normal(0,0.05,(2000,dim)),0,1)
    candidates = np.vstack([candidates, local])

    mu, sigma = gp.predict(candidates, return_std=True)
    ucb = mu + 2*sigma

    nn_pred = nn.predict(candidates)

    if use_svm:
        svm_prob = svm.predict_proba(candidates)[:,1]
    else:
        svm_prob = np.ones(len(candidates))

    score = 0.4*ucb + 0.3*nn_pred + 0.3*svm_prob

    best_idx = np.argmax(score)
    new_point = candidates[best_idx]

    query = "-".join([f"{v:.6f}" for v in new_point])

    print("Submission:", query)

    results.append({"Function": i+1, "Query": query})

pd.DataFrame(results).to_csv("Week4_Submissions.csv", index=False)

print("\nDone ✅")


Function 1
Submission: 0.010526-0.990374

Function 2
Submission: 0.004772-0.978947

Function 3
Submission: 0.680118-0.995208-0.002648

Function 4


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Submission: 0.408484-0.359246-0.432464-0.412663

Function 5


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


Submission: 0.881524-0.985435-0.920544-0.958662

Function 6
Submission: 0.375414-0.156148-0.562724-0.727964-0.019886

Function 7
Submission: 0.009449-0.344044-0.048163-0.336068-0.352414-0.778946

Function 8
Submission: 0.008779-0.537399-0.010010-0.926877-0.789609-0.901063-0.080608-0.060920

Done ✅


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
